Imagine your ecommerce company originally sends: order_id,customer_id,order_status,order_amount,updated_at. Later, the source system introductes a sixth column: order_id,customer_id,order_status,order_amount,updated_at,delivery_type. We need to decide how our pipeline handles delivery_type without losing records or breaking existing processes. Databricks Auto Loader supports multiple approaches, including rescue and addNewColumns

Environment isolation is implemented by source_path = f"{base}/incoming_step5" where we are creating a separate source directory and using a new checkpoint and table. This is useful when testing changes to ingestion logic without modifying existing production pipelines

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

base = "/Volumes/workspace/incremental_project/source_files"

original_path = f"{base}/incoming_clean"
source_path = f"{base}/incoming_step5"
checkpoint_path = f"{base}/_checkpoints/orders_step5"
schema_path = f"{base}/_schemas/orders_step5"

table_name = "workspace.incremental_project.orders_bronze_step5"

assert not spark.catalog.tableExists(table_name), (
    "Step 5 table already exists. Do not restart setup."
)

dbutils.fs.mkdirs(source_path)

for batch in ["001", "002"]:
    filename = f"orders_batch_{batch}.csv"
    destination = f"{source_path}/{filename}"

    existing = [f.name for f in dbutils.fs.ls(source_path)]

    if filename not in existing:
        dbutils.fs.cp(
            f"{original_path}/{filename}",
            destination
        )

display(dbutils.fs.ls(source_path))

Explicit schema implemented by StructType([...]) defines the expected fields. 

In [ ]:
order_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_amount", StringType(), True),
    StructField("updated_at", StringType(), True),
    StructField("_rescued_data", StringType(), True)
])

.schema(order_schema) defines expected columns and concept is related to schema enforcement
schemaEvolutionMode="rescue" captures unexpected fields and concept is related to schema handling
rescuedDataColumn stores unexpected values and concept is related to data preservation
checkpointLocation tracks ingestion progress
availableNow=True processes available files and steps which is incremental execution

In [ ]:
def run_ingestion():
    orders_stream = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .option("rescuedDataColumn", "_rescued_data")
        .option("header", "true")
        .schema(order_schema)
        .load(source_path)
    )

    bronze_stream = (
        orders_stream
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
    )

    query = (
        bronze_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(table_name)
    )

    query.awaitTermination()

In [ ]:
run_ingestion()

print("Bronze rows:", spark.table(table_name).count())

Additive schema change is implemented by adding delivery_type which is an additive schema change that introduces a new column without removing or renaming existing columns

In [ ]:
batch_003 = """order_id,customer_id,order_status,order_amount,updated_at,delivery_type
ORD009,C109,processing,350.00,2026-10-03 10:00:00,express
ORD010,C110,shipped,175.00,2026-10-03 11:00:00,standard
"""

batch_003_path = f"{source_path}/orders_batch_003.csv"

existing = [f.name for f in dbutils.fs.ls(source_path)]

assert "orders_batch_003.csv" not in existing, (
    "Batch 003 already exists. Do not overwrite it."
)

dbutils.fs.put(batch_003_path, batch_003, overwrite=False)

Because we selected rescue mode, Auto Loader should preserve the new field inside _rescued_data rather than add delivery_type as a normal column. You'll see a JSON value similar to: {"delivery_type":"express","_rescued_data_file_path":"..."}
.option("cloudFiles.schemaEvolutionMode","rescue") instructs Auto Loader to capture unexpected columns in rescued data without evolving declared schema

In [ ]:
run_ingestion()

bronze_df = spark.table(table_name)

print("Total rows:", bronze_df.count())

display(
    bronze_df
    .filter(F.col("order_id").isin("ORD009", "ORD010"))
    .select("order_id", "_rescued_data")
)

Shema adaptation is implemented at F.get_json_object(
    F.col("_rescued_data"),
    "$.delivery_type"
) which extracts delivery_type from rescued JSON using a JSON path. For older orders, value will be null because those source files didn't have the column

In [ ]:
evolved_df = (
    bronze_df
    .withColumn(
        "delivery_type",
        F.get_json_object(
            F.col("_rescued_data"),
            "$.delivery_type"
        )
    )
)
display(
    evolved_df.select(
        "order_id",
        "delivery_type"
    ).orderBy("order_id")
)

In [ ]:
evolved_table = (
    "workspace.incremental_project.orders_bronze_step5_evolved"
)

# Create the table only if it doesn't already exist
if not spark.catalog.tableExists(evolved_table):

    (
        evolved_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(evolved_table)
    )

    print("Evolved table created successfully!")

else:
    print("Table already exists. Skipping creation.")

In [ ]:
result = spark.table(evolved_table)

assert result.count() == 12

assert "delivery_type" in result.columns

values = {
    row["order_id"]: row["delivery_type"]
    for row in result
        .filter(F.col("order_id").isin("ORD009", "ORD010"))
        .select("order_id", "delivery_type")
        .collect()
}

assert values["ORD009"] == "express"
assert values["ORD010"] == "standard"

print("Step 5 validation passed!")

Schema evolution means changing the data structure over time. Rescue mode stores unexpected values. 